# SHIPP Stage 8B — Vision enrichment → Bronze raw responses

Primary image per listing (`silver_listing_files`) → vision endpoint → `shipp_bronze.vision_responses` (append-only)

| Decision | Value (all in `config/settings.py`) |
|---|---|
| Model | `VISION_ENDPOINT` — Databricks Model Serving chat endpoint with image input |
| Prompt | `rag/vision.py`, versioned by `VISION_PROMPT_VERSION` |
| Output contract | `{"image_description": str, "detected_labels": [str]}` — anything else is a FAILED row |
| Cache | skip (file, sha256, endpoint, prompt version) already OK or failed non-retryably |
| Retry | 429/5xx/network only, backoff, honours `Retry-After`; 4xx and bad input never retried |
| Raw preservation | every call's raw body or error is appended to Bronze **before** any parsing |
| Never invented | no description → no attributes; the listing stays searchable by its text |

**Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert. Only writer of `vision_responses`.

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
# Reload the rag modules so edits to rag/*.py always take effect (same rule as shipp_silver_lib).
for _name in ["rag.text", "rag.vision", "rag.content", "rag.search_docs", "rag.index",
              "rag.retrieval", "rag.semantic", "rag.evaluation"]:
    importlib.reload(importlib.import_module(_name))

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("RAG modules reloaded from", REPO_ROOT)

## Step 1 — Endpoint check + Bronze table

In [ ]:
try:
    _ep = w.serving_endpoints.get(VISION_ENDPOINT)
    print("Vision endpoint:", _ep.name, "| ready:", _ep.state.ready if _ep.state else None)
except Exception as e:
    _names = sorted(s.name for s in w.serving_endpoints.list())
    raise AssertionError(
        f"FAILED: vision endpoint {VISION_ENDPOINT!r} not found ({type(e).__name__}). "
        f"Set VISION_ENDPOINT in config/settings.py to one of: {_names}"
    )

In [ ]:
import hashlib
import uuid
from datetime import datetime, timezone

from rag import vision
from rag.content import primary_image_files

spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE_VISION_RESPONSES} ({vision.BRONZE_VISION_SCHEMA}) USING DELTA")
client = vision.VisionClient.from_workspace(VISION_ENDPOINT)
print(client, "→", BRONZE_VISION_RESPONSES, "| prompt", VISION_PROMPT_VERSION)

## Step 2 — Images not processed yet for this endpoint + prompt version

In [ ]:
primary = primary_image_files(spark.table(SILVER_LISTING_FILES)).filter(F.col("file_exists"))
done = (
    spark.table(BRONZE_VISION_RESPONSES)
    .filter((F.col("vision_endpoint") == VISION_ENDPOINT) & (F.col("prompt_version") == VISION_PROMPT_VERSION))
    .filter(((F.col("http_status") == 200) & F.col("error_message").isNull()) | ~F.col("retryable"))
    .select("listing_file_id", "file_sha256").distinct()
)
todo = primary.join(done, ["listing_file_id", "file_sha256"], "left_anti").orderBy("listing_id").collect()
print(f"Primary images: {primary.count()} | already done: {primary.count() - len(todo)} | to call: {len(todo)}"
      f" | cap per run: {VISION_MAX_CALLS_PER_RUN}")

## Step 3 — Call the vision endpoint and append every raw result to Bronze

In [ ]:
bronze_rows, ok_calls, failed_calls = [], 0, 0
try:
    for n, f in enumerate(todo[:VISION_MAX_CALLS_PER_RUN], 1):
        mime = vision.mime_type_for(f["file_path"])
        try:
            with open(f["file_path"], "rb") as fh:
                data = fh.read()
            sha = hashlib.sha256(data).hexdigest()
            result = client.describe_image(data, mime, sha)
        except OSError as e:  # file vanished between notebook 50 and now: record it, retry next run
            data, sha = b"", f["file_sha256"]
            result = vision.VisionResult(json.dumps({"endpoint": VISION_ENDPOINT}), None, None,
                                         f"file unreadable: {type(e).__name__}", True, 0, 0)
        bronze_rows.append((
            str(uuid.uuid4()), datetime.now(timezone.utc).replace(tzinfo=None),
            f["listing_id"], f["listing_file_id"], f["file_path"], sha, len(data), mime,
            VISION_ENDPOINT, VISION_PROMPT_VERSION, result.request_meta_json, result.http_status,
            result.response_body, result.error_message, result.retryable, result.attempts, result.latency_ms,
        ))
        if result.ok:
            ok_calls += 1
        else:
            failed_calls += 1
            print(f"  {f['listing_id']} / {f['listing_file_id']} FAILED: {result.error_message}")
        time.sleep(VISION_MIN_SECONDS_BETWEEN_CALLS)
finally:
    if bronze_rows:  # persisted even if the loop is interrupted
        (spark.createDataFrame(bronze_rows, vision.BRONZE_VISION_SCHEMA)
         .write.format("delta").mode("append").saveAsTable(BRONZE_VISION_RESPONSES))
    print(f"Appended {len(bronze_rows)} raw responses (ok={ok_calls}, failed={failed_calls})")

remaining = max(0, len(todo) - VISION_MAX_CALLS_PER_RUN)
if remaining:
    print(f"NOTE — {remaining} image(s) left for the next run.")

## Step 4 — Gate + evidence (raw JSON and what it parses to)

In [ ]:
assert not todo or ok_calls > 0, (
    "FAILED: every vision call failed — read error_message in Bronze (endpoint name, permissions, image format)."
)
recent = (spark.table(BRONZE_VISION_RESPONSES)
          .select("called_at", "listing_id", "listing_file_id", "http_status", "attempts", "latency_ms",
                  "retryable", "error_message", F.substring("response_body", 1, 300).alias("response_preview"))
          .orderBy(F.col("called_at").desc()).limit(20))
display(recent)

example = (spark.table(BRONZE_VISION_RESPONSES)
           .filter((F.col("http_status") == 200) & F.col("error_message").isNull())
           .orderBy(F.col("called_at").desc()).select("listing_id", "response_body").first())
if example:
    print("Parsed example for", example["listing_id"], "→", vision.parse_vision_output(example["response_body"]))

summary = {"table": BRONZE_VISION_RESPONSES, "calls_ok": ok_calls, "calls_failed": failed_calls,
           "calls_remaining": remaining, "endpoint": VISION_ENDPOINT, "prompt": VISION_PROMPT_VERSION,
           "status": "PASS"}
print(summary)
dbutils.notebook.exit(json.dumps(summary))